# Agent 健壮性动手实践
## 2026-05-08 | 第1周周五 | 错误重试 + 工具结果验证 + 降级策略

> **学习目标**: 给 Agent 装上"保险丝"——从 Demo 级提升到生产级的三个弹性层次

**你将学到：**
- 工具调用为什么不可靠？（网络抖动、API 超时、格式异常、模型幻觉）
- 三层防御体系：重试（Retry）→ 熔断（Circuit Breaker）→ 降级（Fallback）
- 指数退避（Exponential Backoff）+ 随机抖动（Jitter）的数学原理
- 工具结果校验——防止垃圾数据污染 LLM 上下文
- 混沌工程（Chaos Engineering）——主动注入故障验证弹性

**架构演进：**
```
前天(FC):  用户 → Agent Loop → LLM → tool_calls → 裸调用工具 → 回传
昨天(MCP): 用户 → Agent Loop → LLM → tool_calls → MCP Client/Server → 回传
今天(弹性):用户 → Agent Loop → LLM → tool_calls → [重试→校验→熔断→降级] → 回传
```

> **参考**: Anthropic《Writing Effective Tools for Agents》: https://www.anthropic.com/engineering/writing-tools-for-agents

---

## 1. 环境准备

In [1]:
import asyncio, json, math, os, random, sys, time
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

llm_client = AsyncOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.deepseek.com",
)

print("环境就绪!")
print(f"  AsyncOpenAI 客户端已初始化")
print(f"  今日主题: 健壮性三重防护")

环境就绪!
  AsyncOpenAI 客户端已初始化
  今日主题: 健壮性三重防护


## 2. 为什么 Agent 需要弹性设计？（八股题 14）

### 工具调用不是 100% 可靠的

Agent 工具调用的每一个环节都可能出错：

| 环节 | 典型故障 | 概率 |
|------|----------|------|
| **网络层** | DNS 解析失败、连接超时、连接被重置 | 经常 |
| **服务层** | 5xx 错误、限流(429)、服务降级 | 偶尔 |
| **数据层** | 返回格式变更、字段缺失、编码错误 | 时不时 |
| **模型层** | 生成错误的函数参数、幻觉出不存在的工具 | 偶发 |

### 不加保护会发生什么？

```
用户: "北京天气怎么样？"
Agent → LLM → tool_calls → get_weather("北京")
                              ↓
                    [连接超时 TimeoutError]
                              ↓
              Agent 收到异常 → 崩溃退出
```

用户的体验：**Agent 直接报错，没有任何解释，也没有任何兜底。**

加上弹性后：
```
用户: "北京天气怎么样？"
Agent → LLM → tool_calls → get_weather("北京")
                              ↓ 第1次：超时
                              ↓ 等待 0.3s，重试
                              ↓ 第2次：成功！
                              ↓ 校验结果格式 ✓
                              ↓ 返回给 LLM
Agent: "北京今天 22°C，晴天……"
```

> **面试要点**: Agent 的可靠性 = 工具可靠性 × LLM 可靠性 × 循环逻辑可靠性。
> 任何一个环节的不稳定性都会被放大。弹性设计就是给每个环节加保护壳。

### 2.1 错误分类 —— 什么该重试，什么不该重试？

这是弹性设计的**第一性原理**：不同错误需要不同的处理策略。

| 错误类型 | 示例 | 是否可重试 | 策略 |
|----------|------|-----------|------|
| **瞬态错误** (Transient) | 连接超时、503、限流 | ✅ 重试 | 指数退避重试 |
| **输入错误** (Input) | 无效表达式、城市名拼错 | ❌ 不重试 | 返回错误描述给 LLM |
| **逻辑错误** (Logic) | 除零、参数类型错误 | ❌ 不重试 | 返回错误描述给 LLM |
| **熔断错误** (Circuit) | 连续失败超过阈值 | ❌ 不重试 | 降级返回 fallback |

**核心判断法则**：如果同样的输入、同样的操作，等待一段时间后可能成功 → 值得重试。
如果同样的输入、同样的操作，重试 100 次也会失败 → 不重试，直接返回错误。

## 3. 弹性配置 —— 所有参数集中管理

In [2]:
# ═══ 弹性配置 —— 生产环境可调参数 ═══

RESILIENCE_CONFIG = {
    # 重试策略
    "max_retries": 3,          # 最大重试次数
    "base_delay": 0.3,         # 基础等待秒数（指数退避：delay × 2^attempt）
    "max_delay": 5.0,          # 单次等待上限
    "retryable_errors": (      # 可重试的错误类型关键字
        "timeout", "connection", "rate_limit", "server_error",
        "server error", "internal", "temporary", "unavailable",
    ),

    # 熔断器
    "circuit_breaker_threshold": 3,   # 连续失败 N 次后熔断
    "circuit_breaker_reset": 15,      # 熔断后 N 秒尝试半开

    # 超时
    "tool_timeout": 8,         # 单次工具调用超时秒数

    # 混沌模式（仅用于演示/测试）
    "chaos_enabled": True,    # 是否注入随机故障
    "chaos_fail_rate": 0.25,   # 随机故障概率
    "chaos_timeout_rate": 0.1, # 随机超时概率
    "chaos_corrupt_rate": 0.1, # 随机返回格式损坏概率
}

print("弹性配置已加载")
print(f"  重试: 最多 {RESILIENCE_CONFIG['max_retries']} 次，基础延迟 {RESILIENCE_CONFIG['base_delay']}s")
print(f"  熔断: 连续 {RESILIENCE_CONFIG['circuit_breaker_threshold']} 次失败触发")
print(f"  超时: 单次调用 {RESILIENCE_CONFIG['tool_timeout']}s")

弹性配置已加载
  重试: 最多 3 次，基础延迟 0.3s
  熔断: 连续 3 次失败触发
  超时: 单次调用 8s


## 4. 工具实现（与前两天完全相同）

> **关键认知**: 弹性层是"包裹"在工具外部的保护壳。工具本身不需要修改——
> 就像给裸电路加装保险丝和稳压器，电路本身的设计不变。

In [3]:
# ── 天气查询（与前两天完全一致）──
def get_weather(city: str, unit: str = "celsius") -> dict:
    weather_db = {
        "北京":    {"temp_c": 22, "condition": "晴",     "humidity": 40, "wind": "北风 3级"},
        "上海":    {"temp_c": 25, "condition": "多云",   "humidity": 68, "wind": "东南风 2级"},
        "广州":    {"temp_c": 29, "condition": "雷阵雨", "humidity": 85, "wind": "南风 4级"},
        "深圳":    {"temp_c": 28, "condition": "阴",     "humidity": 78, "wind": "东风 3级"},
        "杭州":    {"temp_c": 24, "condition": "小雨",   "humidity": 72, "wind": "东北风 2级"},
        "成都":    {"temp_c": 21, "condition": "阴",     "humidity": 75, "wind": "无持续风向 1级"},
        "武汉":    {"temp_c": 26, "condition": "多云",   "humidity": 62, "wind": "南风 2级"},
        "tokyo":   {"temp_c": 18, "condition": "晴",     "humidity": 50, "wind": "北风 2级"},
        "london":  {"temp_c": 13, "condition": "小雨",   "humidity": 80, "wind": "西风 5级"},
        "new york":{"temp_c": 16, "condition": "多云",   "humidity": 55, "wind": "西南风 4级"},
        "sydney":  {"temp_c": 20, "condition": "晴",     "humidity": 45, "wind": "东风 3级"},
        "paris":   {"temp_c": 15, "condition": "阴",     "humidity": 70, "wind": "西南风 3级"},
    }
    key = city.strip().lower()
    data = weather_db.get(key, {"temp_c": 20, "condition": "暂无数据", "humidity": 60, "wind": "未知"})
    temp = data["temp_c"]
    unit_label = "°C"
    if unit == "fahrenheit":
        temp = round(temp * 9/5 + 32, 1)
        unit_label = "°F"
    return {
        "city": city, "temperature": temp, "unit": unit_label,
        "condition": data["condition"], "humidity": f"{data['humidity']}%", "wind": data["wind"]
    }

# ── 计算器（与前两天完全一致）──
def calculate(expression: str) -> dict:
    allowed = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
    allowed.update({"abs": abs, "round": round, "min": min, "max": max, "pow": pow, "sum": sum})
    try:
        result = eval(expression, {"__builtins__": {}}, allowed)
        return {"expression": expression, "result": result, "error": None}
    except Exception as e:
        return {"expression": expression, "result": None, "error": str(e)}

TOOL_EXECUTORS = {"get_weather": get_weather, "calculate": calculate}

print("工具函数就绪！")
print(f"  get_weather('北京'): {json.dumps(get_weather('北京'), ensure_ascii=False)}")
print(f"  calculate('sqrt(144)'): {json.dumps(calculate('sqrt(144)'), ensure_ascii=False)}")

工具函数就绪！
  get_weather('北京'): {"city": "北京", "temperature": 22, "unit": "°C", "condition": "晴", "humidity": "40%", "wind": "北风 3级"}
  calculate('sqrt(144)'): {"expression": "sqrt(144)", "result": 12.0, "error": null}


## 5. 第一层防御：重试（Retry）+ 指数退避

### 为什么不能简单 while 循环重试？

如果 1000 个 Agent 同时遇到超时，同时重试 → 服务瞬间被打爆 → **"雷鸣羊群效应"**（Thundering Herd）。

### 指数退避 (Exponential Backoff)

```
第1次重试: 等待 0.3s  （base_delay × 2^0）
第2次重试: 等待 0.6s  （base_delay × 2^1）
第3次重试: 等待 1.2s  （base_delay × 2^2）
```

### 随机抖动 (Jitter)

在等待时间上叠加随机值（±20%），避免多个 client 在同一个时间点"同步醒来"。

```
实际等待 = delay + random(0, delay × 0.2)
```

In [4]:
# ─── 5.1 可重试错误判断 ───

def is_retryable(error_message: str) -> bool:
    # 判断错误是否属于瞬态错误（值得重试）。
    # 核心原则：只重试「可能自己恢复」的错误，不重试「注定失败」的错误。
    # timeout / 502 → 可重试 ✓  |  invalid expression → 不可重试 ✗
    msg_lower = error_message.lower()
    return any(kw in msg_lower for kw in RESILIENCE_CONFIG["retryable_errors"])

# 快速测试
print("timeout →", is_retryable("connection timeout after 30s"))
print("503 →", is_retryable("503 Service Temporarily Unavailable"))
print("rate limit →", is_retryable("rate_limit exceeded"))
print("invalid input →", is_retryable("invalid expression: syntax error"))
print("division by zero →", is_retryable("division by zero"))

timeout → True
503 → True
rate limit → True
invalid input → False
division by zero → False


### 5.2 重试逻辑演示

In [6]:
async def demo_retry_with_backoff():
    # 演示带指数退避和抖动的重试机制。
    cfg = RESILIENCE_CONFIG
    for attempt in range(cfg["max_retries"] + 1):
        if attempt == 0:
            print(f"[初始调用] 尝试第 0 次")
        else:
            delay = min(cfg["base_delay"] * (2 ** (attempt - 1)), cfg["max_delay"])
            jitter = delay * 0.2 * random.random()
            wait = delay + jitter
            print(f"[第 {attempt} 次重试] 等待 {wait:.3f}s "
                  f"(base={delay:.3f}s + jitter={jitter:.3f}s)")
            await asyncio.sleep(0.05)  # 演示用，实际会真的等待

        # 模拟：前2次失败，第3次成功
        if attempt < 2:
            print(f"  → 失败: connection timeout")
        else:
            print(f"  → 成功!")
            break

await demo_retry_with_backoff()

[初始调用] 尝试第 0 次
  → 失败: connection timeout
[第 1 次重试] 等待 0.334s (base=0.300s + jitter=0.034s)
  → 失败: connection timeout
[第 2 次重试] 等待 0.662s (base=0.600s + jitter=0.062s)
  → 成功!


## 6. 第二层防御：工具结果校验

### 为什么要校验工具返回结果？

LLM 会**无条件信任**工具返回的内容，并基于它组织回复：

```
工具返回: {"temperature": "ERROR"}  ← 格式错误！
LLM回复: "北京当前温度是 ERROR 度……"   ← LLM 照单全收！
```

更危险的情况——LLM 看到残缺数据后**脑补**：
```
工具返回: {"city": "北京"}  ← 缺少 temperature 字段
LLM回复: "北京今天 22°C，晴天……"  ← LLM 编造了不存在的数据！
```

### 校验策略

| 检查项 | 天气 | 计算 |
|--------|------|------|
| 必填字段存在 | city, temperature, condition | expression, result |
| 字段类型正确 | temperature 是数字 | result 是数字 |
| 值域合理 | -90~60°C（地球表面温度） | 非 NaN, 非 Infinity |

In [8]:
# ─── 6.1 天气结果校验 ───
def validate_weather_result(result: dict) -> "tuple[bool, str]":
    required_fields = ["city", "temperature", "condition"]
    for field in required_fields:
        if field not in result:
            return False, f"缺少必填字段: {field}"
    if not isinstance(result.get("temperature"), (int, float)):
        return False, f"temperature 类型异常: {type(result.get('temperature'))}"
    if result.get("temperature", 0) < -90 or result.get("temperature", 0) > 60:
        return False, f"temperature 值异常: {result['temperature']}（合理范围 -90~60°C）"
    return True, ""

# ─── 6.2 计算结果校验 ───
def validate_calculate_result(result: dict) -> "tuple[bool, str]":
    if "result" not in result:
        return False, "缺少必填字段: result"
    r = result["result"]
    if r is not None:
        if not isinstance(r, (int, float)):
            return False, f"result 类型异常: {type(r)}"
        if isinstance(r, float) and (math.isnan(r) or math.isinf(r)):
            return False, f"result 值为 NaN 或 Infinity"
    if "expression" not in result:
        return False, "缺少必填字段: expression"
    return True, ""

RESULT_VALIDATORS = {
    "get_weather": validate_weather_result,
    "calculate": validate_calculate_result,
}

# 快速测试：正常 vs 异常
print("✓ 正常天气:", validate_weather_result({"city": "北京", "temperature": 22, "condition": "晴"}))
print("✗ 缺少字段:", validate_weather_result({"city": "北京"}))
print("✗ 值域异常:", validate_weather_result({"city": "北京", "temperature": 999, "condition": "晴"}))
print("✗ 类型错误:", validate_weather_result({"city": "北京", "temperature": "ERROR", "condition": "晴"}))
print()
print("✓ 正常计算:", validate_calculate_result({"expression": "1+1", "result": 2}))
print("✗ NaN:", validate_calculate_result({"expression": "0/0", "result": float("nan")}))

✓ 正常天气: (True, '')
✗ 缺少字段: (False, '缺少必填字段: temperature')
✗ 值域异常: (False, 'temperature 值异常: 999（合理范围 -90~60°C）')
✗ 类型错误: (False, "temperature 类型异常: <class 'str'>")

✓ 正常计算: (True, '')
✗ NaN: (False, 'result 值为 NaN 或 Infinity')


## 7. 第三层防御：熔断器（Circuit Breaker）

### 为什么需要熔断？

如果重试了 3 次都失败了，说明服务很可能**真的坏了**。继续重试只会：
- 浪费 Agent 的响应时间（用户傻等）
- 给已经崩溃的服务施加更多压力
- 消耗 API 额度（每次重试都会产生 LLM 调用）

熔断器的解决方案：**快速失败（Fail Fast）**——既然知道它会失败，就不要再试了。

### 三个状态

```
  ┌─────────┐    连续失败 ≥ 阈值    ┌─────────┐
  │ CLOSED  │ ───────────────────→ │  OPEN   │
  │ (正常)  │                      │ (熔断)  │
  └─────────┘                      └────┬────┘
       ↑                                │
       │         重置超时后              │
       │     ┌───────────┐              │
       └──── │ HALF_OPEN │ ←───────────┘
             │  (探测)   │
             └───────────┘
          成功→CLOSED / 失败→OPEN
```

> 这是分布式系统容错的经典模式，最早由 Michael Nygard 在《Release It!》(2007) 中系统阐述。

In [9]:
class CircuitBreaker:
    # 熔断器——防止对已经失败的服务持续发起请求。

    def __init__(self, threshold: int, reset_timeout: float):
        self.threshold = threshold
        self.reset_timeout = reset_timeout
        self.failure_count = 0
        self.last_failure_time = 0.0
        self.state = "CLOSED"  # CLOSED | OPEN | HALF_OPEN

    def allow_request(self) -> bool:
        now = time.time()
        if self.state == "CLOSED":
            return True
        if self.state == "OPEN":
            if now - self.last_failure_time >= self.reset_timeout:
                self.state = "HALF_OPEN"
                return True  # 允许一次探测
            return False
        return True  # HALF_OPEN: 允许一次探测

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.threshold:
            self.state = "OPEN"

    def status(self) -> str:
        return f"[状态:{self.state} | 连续失败:{self.failure_count}/{self.threshold}]"


CIRCUIT_BREAKERS = {
    name: CircuitBreaker(
        threshold=RESILIENCE_CONFIG["circuit_breaker_threshold"],
        reset_timeout=RESILIENCE_CONFIG["circuit_breaker_reset"],
    ) for name in TOOL_EXECUTORS
}

# 演示熔断器状态转换
demo_cb = CircuitBreaker(threshold=2, reset_timeout=0.5)
print(f"初始状态: {demo_cb.status()}")
demo_cb.record_failure()
print(f"失败1次:  {demo_cb.status()}")
print(f"允许请求? {demo_cb.allow_request()}")
demo_cb.record_failure()
print(f"失败2次:  {demo_cb.status()}")
print(f"允许请求? {demo_cb.allow_request()}  ← 熔断！拒绝请求")
print()
print("等待 reset_timeout 后...")
await asyncio.sleep(0.6)
print(f"当前状态: {demo_cb.status()}")
print(f"允许请求? {demo_cb.allow_request()}  ← 半开状态，允许一次探测")
demo_cb.record_success()
print(f"探测成功:  {demo_cb.status()}  ← 恢复到正常")

初始状态: [状态:CLOSED | 连续失败:0/2]
失败1次:  [状态:CLOSED | 连续失败:1/2]
允许请求? True
失败2次:  [状态:OPEN | 连续失败:2/2]
允许请求? False  ← 熔断！拒绝请求

等待 reset_timeout 后...
当前状态: [状态:OPEN | 连续失败:2/2]
允许请求? True  ← 半开状态，允许一次探测
探测成功:  [状态:CLOSED | 连续失败:0/2]  ← 恢复到正常


## 8. 降级策略（Fallback）

当所有重试都失败、熔断器断开时，Agent 不能"死掉"——它需要给用户一个有意义的交代。

### 降级层次

| 优先级 | 策略 | 示例 |
|--------|------|------|
| 1st | **正常结果** | `{"city": "北京", "temperature": 22, ...}` |
| 2nd | **缓存数据** | 返回上次成功的缓存（标注时间戳） |
| 3rd | **默认值** | `{"temperature": "N/A", "condition": "数据暂时不可用"}` |
| 4th | **错误说明** | `{"error": "天气服务暂不可用，请稍后重试"}` |

> 今天实现的是第 3+4 层的组合：返回带有明确 `_fallback` 标记的降级数据，让 LLM 如实告知用户。

In [10]:
FALLBACK_RESULTS = {
    "get_weather": {
        "city": "未知",
        "temperature": "N/A",
        "unit": "°C",
        "condition": "天气数据暂时不可用",
        "humidity": "N/A",
        "wind": "N/A",
        "_fallback": True,  # 标记：这是降级数据
    },
    "calculate": {
        "expression": "N/A",
        "result": None,
        "error": "计算服务暂时不可用，请稍后重试",
        "_fallback": True,
    },
}

print("降级数据模板已加载")
print(f"  天气 Fallback: {json.dumps(FALLBACK_RESULTS['get_weather'], ensure_ascii=False)}")
print(f"  计算 Fallback: {json.dumps(FALLBACK_RESULTS['calculate'], ensure_ascii=False)}")

降级数据模板已加载
  天气 Fallback: {"city": "未知", "temperature": "N/A", "unit": "°C", "condition": "天气数据暂时不可用", "humidity": "N/A", "wind": "N/A", "_fallback": true}
  计算 Fallback: {"expression": "N/A", "result": null, "error": "计算服务暂时不可用，请稍后重试", "_fallback": true}


## 9. 混沌工程（Chaos Engineering）—— 主动注入故障

### 怎么验证弹性层真的有效？

等故障自然发生 → 可能等几个月 → 验证周期太长。

混沌工程的思路：**主动注入故障**，在受控环境下观察系统的弹性表现。

### 三种注入模式

| 模式 | 模拟场景 | 触发弹性机制 |
|------|----------|------------|
| **超时注入** | 第三方 API hang 住 | 重试 + 熔断 |
| **故障注入** | 服务端 500/502/限流 | 重试 + 熔断 |
| **格式损坏** | API 返回格式变更 | 结果校验 + 熔断 |

In [11]:
class ChaosException(Exception):
    # 混沌注入异常——仅用于测试/演示。
    pass


def chaos_inject(tool_name: str):
    # 根据混沌配置注入随机故障。返回 "corrupt" 表示格式损坏。
    cfg = RESILIENCE_CONFIG
    if not cfg["chaos_enabled"]:
        return None  # 无注入

    roll = random.random()

    if roll < cfg["chaos_timeout_rate"]:
        raise ChaosException(f"[混沌] {tool_name} 模拟超时")
    elif roll < cfg["chaos_timeout_rate"] + cfg["chaos_fail_rate"]:
        msgs = ["connection reset", "500 Internal Server Error",
                "rate_limit exceeded", "service temporarily unavailable"]
        raise ChaosException(f"[混沌] {tool_name}: {random.choice(msgs)}")
    elif roll < cfg["chaos_timeout_rate"] + cfg["chaos_fail_rate"] + cfg["chaos_corrupt_rate"]:
        return "corrupt"  # 模拟格式损坏
    return None

print("混沌注入器已就绪（当前关闭）")
print("  启用方式: RESILIENCE_CONFIG['chaos_enabled'] = True")

混沌注入器已就绪（当前关闭）
  启用方式: RESILIENCE_CONFIG['chaos_enabled'] = True


## 10. 整合所有层：弹性工具执行器

这是今天**最核心的代码**——把重试、校验、熔断、降级四个组件整合为一个统一的工具调度入口。

```
resilient_execute_tool(name, args)
  │
  ├─ 0. 熔断器检查 ──→ 已熔断? → 直接返回降级数据
  │
  ├─ 1. 重试循环 (最多 N 次)
  │    ├─ 混沌注入（演示模式）
  │    ├─ 执行工具 + 超时控制
  │    ├─ 结果校验
  │    ├─ 成功 → 记录到熔断器 → 返回
  │    └─ 失败 → 判断可重试?
  │         ├─ 是 → 指数退避等待 → 继续循环
  │         └─ 否 → 退出循环
  │
  └─ 2. 全部失败 → 返回降级数据
```

> **关键认知**: 这个函数是「非侵入式」的——Agent 循环只需要把原来的
> `execute_tool()` 替换为 `await resilient_execute_tool()`，其他逻辑完全不变。

In [13]:
async def resilient_execute_tool(
    name: str,
    args: dict,
    verbose: bool = True,
) -> str:
    # 带完整弹性保护的工具执行入口。三层防御：熔断器 → 重试+校验 → 降级
    cb = CIRCUIT_BREAKERS[name]

    # ── 第0步：熔断器检查 ──
    if not cb.allow_request():
        if verbose:
            print(f"  ⚡ {name} 已熔断，直接降级")
        return json.dumps(FALLBACK_RESULTS.get(name, {"error": "服务不可用"}), ensure_ascii=False)

    # ── 第1步：重试循环 ──
    last_error = None
    cfg = RESILIENCE_CONFIG

    for attempt in range(cfg["max_retries"] + 1):
        try:
            # 混沌注入（演示模式）
            chaos_result = chaos_inject(name)
            if chaos_result == "corrupt":
                # 模拟格式损坏
                if name == "get_weather":
                    result = {"city": args.get("city", "?"), "temperature": "ERROR"}
                else:
                    result = {"result": float("nan")}
            elif isinstance(chaos_result, ChaosException):
                raise chaos_result
            else:
                # 实际执行（带超时控制）
                func = TOOL_EXECUTORS[name]
                try:
                    result = await asyncio.wait_for(
                        asyncio.to_thread(func, **args),
                        timeout=cfg["tool_timeout"],
                    )
                except asyncio.TimeoutError:
                    raise ChaosException(f"{name} 执行超时 (>{cfg['tool_timeout']}s)")

            # 结果校验
            validator = RESULT_VALIDATORS.get(name)
            if validator and isinstance(result, dict):
                is_valid, err_msg = validator(result)
                if not is_valid:
                    raise ChaosException(f"{name} 格式校验失败: {err_msg}")

            # 成功！
            cb.record_success()
            if verbose and attempt > 0:
                print(f"  ✓ {name} 重试成功（第 {attempt} 次）")
            return json.dumps(result, ensure_ascii=False)

        except ChaosException as e:
            last_error = str(e)
        except Exception as e:
            last_error = f"未知异常: {type(e).__name__}: {e}"

        cb.record_failure()

        # 判断是否值得重试
        if not is_retryable(last_error):
            if verbose:
                print(f"  ✗ {name} 不可重试，放弃: {last_error[:80]}")
            break

        if attempt < cfg["max_retries"]:
            delay = min(cfg["base_delay"] * (2 ** attempt), cfg["max_delay"])
            jitter = delay * 0.2 * random.random()
            wait = delay + jitter
            if verbose:
                print(f"  ↻ {name} 第 {attempt + 1}/{cfg['max_retries']} 次重试 "
                      f"（等待 {wait:.2f}s）: {last_error[:60]}")
            await asyncio.sleep(wait)

    # ── 第2步：全部失败 → 降级 ──
    if verbose:
        print(f"  ▼ {name} 全部重试失败({cb.status()})，使用降级数据")
    fallback = dict(FALLBACK_RESULTS.get(name, {"error": last_error, "_fallback": True}))
    if name == "get_weather":
        fallback["city"] = args.get("city", "未知")
    return json.dumps(fallback, ensure_ascii=False)

print("弹性工具执行器就绪！")

弹性工具执行器就绪！


## 11. Enhanced System Prompt

> 新增降级数据的处理指令——告诉 LLM 如何识别和应对降级数据。

In [14]:
SYSTEM_PROMPT = (
    "你是一个具备工具调用能力的智能助手。你拥有以下工具：\n\n"
    "1. get_weather — 查询任意城市的实时天气（温度、天气状况、湿度、风速）\n"
    "2. calculate   — 执行数学表达式计算（支持四则运算、幂运算、三角函数等）\n\n"
    "行为准则：\n"
    "- 用户询问天气相关信息时，主动调用 get_weather\n"
    "- 用户需要数值计算时，调用 calculate，禁止自行心算\n"
    "- 收到工具返回结果后，用流畅的中文向用户转述\n"
    '- 如果工具返回中包含 "_fallback": true 或 "暂时不可用"，向用户如实说明数据异常，不要编造数据\n'
    "- 保持回答简洁、信息密度高"
)

print("Enhanced System Prompt 已加载，长度:", len(SYSTEM_PROMPT), "字符")

Enhanced System Prompt 已加载，长度: 284 字符


In [16]:
WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "查询指定城市的实时天气信息。返回数据包含：温度、天气状况、湿度、风速。",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "城市名称，支持中文或英文"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "温度单位"},
            },
            "required": ["city"],
        },
    },
}

CALCULATOR_TOOL = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "执行数学表达式计算。当用户需要精确数值计算时必须调用，禁止心算。",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "数学表达式，如 '(2+3)*4'"},
            },
            "required": ["expression"],
        },
    },
}

TOOLS = [WEATHER_TOOL, CALCULATOR_TOOL]
print("工具 Schema 已加载")

工具 Schema 已加载


## 12. Agent 循环 —— 弹性版

与前两天的 Agent 循环主体**完全一致**，唯一的区别在工具调用那一行：

```python
# 昨天（裸调用）:
result_json = execute_tool(tool_name, tool_args)

# 今天（弹性调用）:
result_json = await resilient_execute_tool(tool_name, tool_args)
```

这就是「非侵入式弹性改造」——工具调度层独立升级，Agent 循环主体零改动。

In [17]:
async def run_agent(
    llm_client: AsyncOpenAI,
    user_query: str,
    model: str = "deepseek-v4-flash",
    max_turns: int = 10,
    verbose: bool = True,
) -> str:
    # Agent 主循环 —— 使用弹性工具调用。
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for turn in range(1, max_turns + 1):
        response = await llm_client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS,
            tool_choice="auto", temperature=0.0,
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            if verbose:
                names = [tc.function.name for tc in msg.tool_calls]
                print(f"\n[轮次 {turn}] LLM 调用工具: {', '.join(names)}")
            messages.append(msg.model_dump())

            for tc in msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)
                if verbose:
                    print(f"  → {tool_name}({json.dumps(tool_args, ensure_ascii=False)})")

                # ★ 关键区别：弹性调用 ★
                result_json = await resilient_execute_tool(tool_name, tool_args, verbose=verbose)

                if verbose:
                    preview = repr(result_json[:100]) + "..." if len(result_json) > 100 else repr(result_json)
                    print(f"  ← {preview}")

                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result_json})
        else:
            if verbose:
                print(f"\n[轮次 {turn}] LLM 最终回复")
            return msg.content

    return "处理超时，请将问题拆分为更小的子问题。"

print("Agent 循环函数已定义（弹性版）")

Agent 循环函数已定义（弹性版）


## 13. 一键查询助手

In [18]:
async def ask(user_query: str, verbose: bool = True) -> str:
    # 弹性 Agent 一键查询助手。
    return await run_agent(llm_client, user_query, verbose=verbose)

print("ask() 助手已就绪！用法: await ask('你的问题')")

ask() 助手已就绪！用法: await ask('你的问题')


## 14. 交互测试

### 14.1 正常模式 — 基础功能验证

先确保正常模式下一切工作正常。

In [19]:
query = "北京今天天气怎么样？适合出去玩吗？"
print(f"用户: {query}")
answer = await ask(query, verbose=True)
print(f"\nAgent 最终回复:\n{answer}")

用户: 北京今天天气怎么样？适合出去玩吗？

[轮次 1] LLM 调用工具: get_weather
  → get_weather({"city": "北京", "unit": "celsius"})
  ✗ get_weather 不可重试，放弃: get_weather 格式校验失败: 缺少必填字段: condition
  ▼ get_weather 全部重试失败([状态:CLOSED | 连续失败:1/3])，使用降级数据
  ← '{"city": "北京", "temperature": "N/A", "unit": "°C", "condition": "天气数据暂时不可用", "humidity": "N/A", "win'...

[轮次 2] LLM 最终回复

Agent 最终回复:
抱歉，目前北京的天气数据暂时查询不到（系统显示数据异常）。😅

无法获取到实时的温度、天气状况、湿度和风速信息，所以我暂时没法帮你判断今天是否适合出去玩。

建议你可以：
- **稍后再查一次**，看看数据能否恢复正常
- 或者打开手机上的天气 App 手动查一下北京的实时天气

如果还有其他我能帮上忙的（比如查其他城市的天气、做计算等），随时告诉我！


### 14.2 正常模式 — 并行调用

In [20]:
query = "上海和广州的天气分别怎么样？"
print(f"用户: {query}")
answer = await ask(query, verbose=True)
print(f"\nAgent 最终回复:\n{answer}")

用户: 上海和广州的天气分别怎么样？

[轮次 1] LLM 调用工具: get_weather, get_weather
  → get_weather({"city": "上海", "unit": "celsius"})
  ← '{"city": "上海", "temperature": 25, "unit": "°C", "condition": "多云", "humidity": "68%", "wind": "东南风 2'...
  → get_weather({"city": "广州", "unit": "celsius"})
  ↻ get_weather 第 1/3 次重试 （等待 0.34s）: [混沌] get_weather: rate_limit exceeded
  ✓ get_weather 重试成功（第 1 次）
  ← '{"city": "广州", "temperature": 29, "unit": "°C", "condition": "雷阵雨", "humidity": "85%", "wind": "南风 4'...

[轮次 2] LLM 最终回复

Agent 最终回复:
以下是上海和广州当前的天气情况：

**🌤 上海**
- **温度：** 25°C
- **天气状况：** 多云
- **湿度：** 68%
- **风速：** 东南风 2级

**⛈ 广州**
- **温度：** 29°C
- **天气状况：** 雷阵雨
- **湿度：** 85%
- **风速：** 南风 4级

广州比上海热不少（高出4°C），而且正在下雷阵雨，湿度较高，出门建议带伞。上海则比较凉爽舒适，多云天气适合户外活动。


### 14.3 正常模式 — 多步推理 + 计算

In [21]:
query = "北京现在多少度？如果北京比成都热5度，成都应该是多少度？"
print(f"用户: {query}")
answer = await ask(query, verbose=True)
print(f"\nAgent 最终回复:\n{answer}")

用户: 北京现在多少度？如果北京比成都热5度，成都应该是多少度？

[轮次 1] LLM 调用工具: get_weather
  → get_weather({"city": "北京", "unit": "celsius"})
  ← '{"city": "北京", "temperature": 22, "unit": "°C", "condition": "晴", "humidity": "40%", "wind": "北风 3级"'...

[轮次 2] LLM 调用工具: calculate
  → calculate({"expression": "22 - 5"})
  ✗ calculate 不可重试，放弃: calculate 格式校验失败: result 值为 NaN 或 Infinity
  ▼ calculate 全部重试失败([状态:CLOSED | 连续失败:1/3])，使用降级数据
  ← '{"expression": "N/A", "result": null, "error": "计算服务暂时不可用，请稍后重试", "_fallback": true}'

[轮次 3] LLM 最终回复

Agent 最终回复:
计算服务暂时不可用，不过这是一个非常简单的减法：**22 - 5 = 17°C**。

所以，如果北京比成都热5度，那么成都应该是 **17°C**。


### 14.4 ⚡ 混沌模式 —— 模拟故障场景

这是今天**最重要的测试**！开启混沌注入后，观察 Agent 如何在故障中存活。

**运行前请思考：**
- Agent 会直接崩溃吗？
- 重试机制会触发吗？会重试几次？
- 如果所有重试都失败，Agent 会怎么回复？
- 你能从输出中看到熔断器状态变化吗？

In [25]:
# 启用混沌模式
RESILIENCE_CONFIG["chaos_enabled"] = True
RESILIENCE_CONFIG["chaos_fail_rate"] = 0.3
RESILIENCE_CONFIG["chaos_timeout_rate"] = 0.15
RESILIENCE_CONFIG["chaos_corrupt_rate"] = 0.1

print("⚠️  混沌模式已启用！")
print(f"   故障率: {RESILIENCE_CONFIG['chaos_fail_rate']:.0%}")
print(f"   超时率: {RESILIENCE_CONFIG['chaos_timeout_rate']:.0%}")
print(f"   损坏率: {RESILIENCE_CONFIG['chaos_corrupt_rate']:.0%}")
print()

query = "北京今天天气怎么样？"
print(f"用户: {query}")

answer = await ask(query, verbose=True)
print(f"\nAgent 最终回复:\n{answer}")

print()
print("── 熔断器状态 ──")
for name, cb in CIRCUIT_BREAKERS.items():
    print(f"  {name}: {cb.status()}")

# 关闭混沌模式
RESILIENCE_CONFIG["chaos_enabled"] = False

⚠️  混沌模式已启用！
   故障率: 30%
   超时率: 15%
   损坏率: 10%

用户: 北京今天天气怎么样？

[轮次 1] LLM 调用工具: get_weather
  → get_weather({"city": "北京", "unit": "celsius"})
  ✗ get_weather 不可重试，放弃: get_weather 格式校验失败: 缺少必填字段: condition
  ▼ get_weather 全部重试失败([状态:CLOSED | 连续失败:1/3])，使用降级数据
  ← '{"city": "北京", "temperature": "N/A", "unit": "°C", "condition": "天气数据暂时不可用", "humidity": "N/A", "win'...

[轮次 2] LLM 最终回复

Agent 最终回复:
抱歉，目前北京的天气数据暂时不可用，返回的接口显示数据异常，暂时无法查到实时的温度、天气状况、湿度和风速等信息。建议稍后再试一下，或者你直接看看窗外感受一下天气 😄

── 熔断器状态 ──
  get_weather: [状态:CLOSED | 连续失败:1/3]
  calculate: [状态:CLOSED | 连续失败:1/3]


### 14.5 混沌模式 —— 多轮测试：观察熔断器跨调用行为

启用混沌后连续测试多个查询，关键观察点：
- 前几个查询如果导致连续失败 → 熔断器断开
- 后续查询直接跳过重试，快速降级
- `_fallback: true` 标记让 LLM 如实告知用户

In [26]:
RESILIENCE_CONFIG["chaos_enabled"] = True
RESILIENCE_CONFIG["chaos_fail_rate"] = 0.35

# 重置所有熔断器
for cb in CIRCUIT_BREAKERS.values():
    cb.failure_count = 0
    cb.state = "CLOSED"

queries = [
    "深圳天气怎么样？",
    "杭州天气怎么样？",
    "武汉天气怎么样？",
    "成都天气怎么样？",
]

for q in queries:
    print(f"\n{'─'*50}")
    print(f"👤 {q}")
    print(f"{'─'*50}")
    answer = await ask(q, verbose=True)
    preview = answer[:200] + "..." if len(answer) > 200 else answer
    print(f"\n🤖 {preview}")
    print(f"\n📊 熔断器: get_weather={CIRCUIT_BREAKERS['get_weather'].status()}")

RESILIENCE_CONFIG["chaos_enabled"] = False
print("\n混沌模式已关闭")


──────────────────────────────────────────────────
👤 深圳天气怎么样？
──────────────────────────────────────────────────

[轮次 1] LLM 调用工具: get_weather
  → get_weather({"city": "深圳", "unit": "celsius"})
  ✗ get_weather 不可重试，放弃: [混沌] get_weather 模拟超时
  ▼ get_weather 全部重试失败([状态:CLOSED | 连续失败:1/3])，使用降级数据
  ← '{"city": "深圳", "temperature": "N/A", "unit": "°C", "condition": "天气数据暂时不可用", "humidity": "N/A", "win'...

[轮次 2] LLM 最终回复

🤖 抱歉，目前深圳的天气数据暂时不可用，可能是数据源出现异常或暂时无法获取到实时信息。建议稍后再查查看，或者直接查看天气 App 获取最新情况。

📊 熔断器: get_weather=[状态:CLOSED | 连续失败:1/3]

──────────────────────────────────────────────────
👤 杭州天气怎么样？
──────────────────────────────────────────────────

[轮次 1] LLM 调用工具: get_weather
  → get_weather({"city": "杭州", "unit": "celsius"})
  ← '{"city": "杭州", "temperature": 24, "unit": "°C", "condition": "小雨", "humidity": "72%", "wind": "东北风 2'...

[轮次 2] LLM 最终回复

🤖 杭州当前天气如下：

- 🌡 **温度**：24°C
- 🌧 **天气状况**：小雨
- 💧 **湿度**：72%
- 🌬 **风力**：东北风 2级

杭州现在正下着小雨，体感应该比较湿润，出门记得带伞哦！

📊 熔断器: get_weather=[状态:CLOSED

### 14.6 自由输入

输入你自己的问题，试试弹性 Agent。

**挑战题目：**
- "帮我算 256 的平方根，同时查一下东京天气" → 观察并行调用 + 弹性处理
- "计算 log(1000) + sin(pi/4)" → 复杂计算
- 开启混沌模式后问任意问题 → 观察弹性机制如何保护 Agent

In [ ]:
# ===== 自由发挥区：输入任何你想问的 =====
query = input("请输入你的问题: ").strip()
if not query:
    query = "帮我算一下 2**10，然后查一下伦敦天气"
    print(f"(使用默认问题): {query}")

print()
answer = await ask(query, verbose=True)
print(f"\nAgent 最终回复:\n{answer}")

## 15. 三层防御体系回顾

### 架构总览

```
                    ┌──────────────────────────────────┐
                    │       resilient_execute_tool()     │
                    │                                    │
用户输入             │  ┌──────────┐  ┌──────────┐      │
  ↓                 │  │ 熔断器   │  │ 重试循环  │      │
Agent Loop          │  │ Circuit  │→ │ 最多3次   │      │
  ↓                 │  │ Breaker  │  │ 指数退避  │      │
LLM 决策            │  └──────────┘  └────┬─────┘      │
  ↓                 │        ↓ 熔断?      ↓             │
tool_calls ─────────│────→ 直接降级    执行工具        │
                    │                    ↓             │
                    │               ┌──────────┐      │
                    │               │ 结果校验  │      │
                    │               │ Validator │      │
                    │               └────┬─────┘      │
                    │                    ↓             │
                    │         成功 → 返回结果          │
                    │         失败 → 判断可重试?       │
                    │            ├ 可重试 → 退避+重试   │
                    │            └ 不可重试 → 降级     │
                    │                    ↓             │
                    │               ┌──────────┐      │
                    │               │ 降级数据  │      │
                    │               │ Fallback  │      │
                    │               └──────────┘      │
                    └──────────────────────────────────┘
```

### 与前两天的对比

| 维度 | 纯 FC (前天) | MCP (昨天) | 弹性 FC (今天) |
|------|-------------|-----------|---------------|
| **工具定义** | 硬编码 | MCP Server 管理 | 硬编码 + 校验 Schema |
| **工具执行** | 裸调用 | MCP Session | 弹性包装调用 |
| **容错能力** | 无（一次失败就炸） | 依赖 MCP 传输层 | 三层防御 |
| **可观测性** | 低 | 低 | 高（熔断器状态/重试日志） |
| **适用场景** | Demo/原型 | 工具治理 | 生产系统 |

### 面试速记

| 概念 | 一句话 |
|------|-----|
| **重试** | 瞬态错误重试，用指数退避+抖动避免羊群效应 |
| **熔断** | 连续失败 N 次后快速失败，定时半开探测恢复 |
| **降级** | 返回有意义的默认值，让 Agent 如实告知用户 |
| **校验** | 防止垃圾数据进入 LLM 上下文，垃圾进垃圾出 |
| **混沌** | 主动注入故障验证弹性——做最坏的打算 |

## 16. 参考资料 & 延伸阅读

| 主题 | 链接 | 说明 |
|------|------|------|
| **Anthropic: Writing Effective Tools** | https://www.anthropic.com/engineering/writing-tools-for-agents | 今天的核心参考——工具设计最佳实践 |
| **Retry Pattern (Azure)** | https://learn.microsoft.com/en-us/azure/architecture/patterns/retry | 重试模式权威文档 |
| **Circuit Breaker Pattern** | https://learn.microsoft.com/en-us/azure/architecture/patterns/circuit-breaker | 熔断器模式权威文档 |
| **Circuit Breaker (Martin Fowler)** | https://martinfowler.com/bliki/CircuitBreaker.html | 熔断器概念起源 |
| **Release It! (Michael Nygard)** | https://pragprog.com/titles/mnee2/release-it-second-edition/ | 系统稳定性圣经 |
| **Chaos Engineering (Netflix)** | https://netflixtechblog.com/tagged/chaos-engineering | Chaos Monkey 等混沌工具 |
| **OpenAI Function Calling** | https://platform.openai.com/docs/guides/function-calling | FC 官方文档 |
| **DeepSeek API** | https://api-docs.deepseek.com/zh-cn/ | DeepSeek API 文档 |

### 下周预告（第2周）

开始引入 LangGraph！把 Agent 的线性循环升级为**状态图（State Graph）**：
- 用图结构管理 Agent 的多分支决策
- 引入 Checkpoint 实现对话历史的持久化
- 支持 Human-in-the-loop 人工干预

参考: https://langchain-ai.github.io/langgraph/tutorials/introduction/

## 17. 核心知识点回顾

### 今天你掌握了什么

| # | 知识点 | 对应位置 |
|---|--------|---------|
| 1 | **错误分类** — 瞬态 vs 输入 vs 逻辑 vs 熔断，不同策略 | Cell 2.1 |
| 2 | **指数退避 + 抖动** — 重试的数学原理，避免羊群效应 | Cell 5 |
| 3 | **工具结果校验** — 必填字段、类型、值域三重检查 | Cell 6 |
| 4 | **熔断器三态** — CLOSED → OPEN → HALF_OPEN 状态机 | Cell 7 |
| 5 | **降级策略** — Fallback 数据让 Agent 优雅失败 | Cell 8 |
| 6 | **混沌工程** — 主动注入故障验证弹性 | Cell 9 |
| 7 | **弹性执行器** — 三层防御整合为统一入口 | Cell 10 |

### 对应八股题

| 题号 | 主题 | 对应 Cell |
|------|------|-----------|
| 题 14 | Agent 健壮性设计 | 全篇（重点 Cell 2, 10, 15） |

### 代码文件

- `exercises/w1d3-resilient-agent/agent.py` — 完整可运行脚本（弹性 Agent + 混沌模式）
- `resilient_agent.ipynb` — 交互式学习 Notebook（当前文件）
- `exercises/w1d2-mcp-server/agent.py` — 昨天 MCP 版本（对比参考）
- `exercises/w1d1-function-calling/agent.py` — 前天纯 FC 版本（对比参考）

### 第一周学习小结

```
第1天(05-06) → 纯 Function Calling Agent （理解 ReAct 循环）
第2天(05-07) → MCP Server 改造           （理解工具标准化）
第3天(05-08) → 弹性 Agent 设计           （理解生产级可靠性）
────────────────────────────────────────────
              你已经掌握了 Agent 的三大核心能力：
              工具调用 → 工具治理 → 工具可靠性
```

---

> 好的 Agent 不是不犯错的 Agent，而是犯了错也能优雅恢复的 Agent。
> 第一周结束了——你已经从零搭建了一个具备生产级韧性的 Agent 系统。
> 下周我们将引入 LangGraph，用状态图管理更复杂的 Agent 工作流。